
## Raw Data Extraction Structuring to Tabular Form - Parsing Script #1

##### Created by Farwa Rizvi, an MSAI student and team member of the BRCA1/2 research project. 

##### This script converts the raw data exported from the 'All of Us V8 BRCA 1/2 Data Extraction and Annonation Integration' scripts into a structured tabular format. Organizing the data in this way improves readability and enables easier data manipulation for downstream analysis.

##### The script contains several sections that demonstrate how the data was parsed and structured at different stages of the project as additional information was incorporated. The first section, “AMR Variants with VAT,” reflects the most up-to-date structure and shows the final parsing approach used after integrating the Variant Annotation Table (VAT) into the dataset.

### AMR Variants with VAT

v8 Exome x v8 VAT

8 New VAT Variables:
- `gvs_max_subpop` · `aa_change` · `transcript` · `gene_symbol`
- `gvs_max_ac` · `gvs_max_sc` · `gvs_max_an` · `gnomad_max_subpop`


In [1]:
import pandas as pd
import numpy as np
import re

FILE_B1 = "C:/Users/knigh/Downloads/2026_B2_CT_variants_amr.csv"

# Space-delimited Hail export; quotechar handles multi-word fields (e.g. clinvar_classification)
df_b1 = pd.read_csv(FILE_B1, sep=" ", quotechar='"')

print(f"Shape: {df_b1.shape}")
print("Columns:", df_b1.columns.tolist())


Shape: (2322, 50)
Columns: ['locus', 'alleles', 'filters', 'a_index', 'was_split', 'variant_qc.gq_stats.mean', 'variant_qc.gq_stats.stdev', 'variant_qc.gq_stats.min', 'variant_qc.gq_stats.max', 'variant_qc.AC', 'variant_qc.AF', 'variant_qc.AN', 'variant_qc.homozygote_count', 'variant_qc.call_rate', 'variant_qc.n_called', 'variant_qc.n_not_called', 'variant_qc.n_filtered', 'variant_qc.n_het', 'variant_qc.n_non_ref', 'variant_qc.het_freq_hwe', 'variant_qc.p_value_hwe', 'variant_qc.p_value_excess_het', 'info.AC', 'info.AF', 'info.AN', 'info.homozygote_count', 'vat.vid', 'vat.contig', 'vat.position', 'vat.consequence', 'vat.clinvar_classification', 'vat.variant_type', 'vat.ref_allele', 'vat.alt_allele', 'vat.gvs_all_ac', 'vat.gvs_all_an', 'vat.gvs_all_af', 'vat.gvs_all_sc', 'vat.dbsnp_rsid', 'vat.revel', 'vat.gvs_max_subpop', 'vat.aa_change', 'vat.transcript', 'vat.gene_symbol', 'vat.gvs_max_ac', 'vat.gvs_max_sc', 'vat.gvs_max_an', 'vat.gnomad_max_subpop', 'vat.is_canonical_transcript', 'v

### Rename VAT columns — strip `vat.` prefix for easier downstream use


In [2]:
# Remove 'vat.' prefix from all VAT columns to match 2023 BRCA2 style
vat_cols = {c: c.replace('vat.', '') for c in df_b1.columns if c.startswith('vat.')}
df_b1 = df_b1.rename(columns=vat_cols)

print([c for c in df_b1.columns if c.startswith('vat.')])   # should be empty
print([c for c in df_b1.columns if c.startswith('vid') or c == 'contig'][:10])


[]
['vid', 'contig', 'vid_alt']


In [ ]:
# Recreate 2023-style qc_* and info_* fields from Hail variant_qc.* and info.*

df_b1['qc_AC'] = df_b1['variant_qc.AC']
df_b1['qc_AF'] = df_b1['variant_qc.AF']
df_b1['qc_AN'] = df_b1['variant_qc.AN']

df_b1['call_rate'] = df_b1['variant_qc.call_rate']
df_b1['n_het']     = df_b1['variant_qc.n_het']

df_b1['info_AC'] = df_b1['info.AC']
df_b1['info_AF'] = df_b1['info.AF']
df_b1['info_AN'] = df_b1['info.AN']

base_cols = [
    'locus', 'alleles',
    'qc_AC', 'qc_AF', 'qc_AN',
    'call_rate', 'n_het',
    'info_AC', 'info_AF', 'info_AN'
]
print(df_b1[base_cols].head(5).to_string())

#Data preview removed to comply with All of Us data user policies.


In [ ]:
# Ensure core VAT numeric fields are numeric
num_cols = [
    'gvs_all_ac', 'gvs_all_an', 'gvs_all_af', 'gvs_all_sc',
    'gvs_max_ac', 'gvs_max_sc', 'gvs_max_an'
]
for col in num_cols:
    if col in df_b1.columns:
        df_b1[col] = pd.to_numeric(df_b1[col], errors='coerce')

# revel to float
if 'revel' in df_b1.columns:
    df_b1['revel'] = pd.to_numeric(df_b1['revel'], errors='coerce')

# String fields for new variables
str_fields = ['gvs_max_subpop', 'aa_change', 'transcript',
              'gene_symbol', 'gnomad_max_subpop']
for col in str_fields:
    if col in df_b1.columns:
        df_b1[col] = df_b1[col].where(df_b1[col].notna(), other=None)

print(df_b1[['vid', 'consequence', 'clinvar_classification', 'revel',
             'gvs_max_subpop', 'aa_change', 'transcript', 'gene_symbol',
             'gvs_max_ac', 'gvs_max_sc', 'gvs_max_an', 'gnomad_max_subpop']].head(8).to_string())

#Data preview removed to comply with All of Us data user policies.


In [ ]:
# Parse aa_change: "ENSP00000350283.3:p.(His1805Arg)" -> protein_id + hgvsp

def split_aa_change(val):
    if pd.isna(val) or val is None:
        return None, None
    parts = str(val).split(":", 1)
    protein_id = parts[0] if len(parts) > 0 else None
    hgvsp      = parts[1] if len(parts) > 1 else None
    return protein_id, hgvsp

if 'aa_change' in df_b1.columns:
    df_b1[['protein_id', 'hgvsp']] = df_b1['aa_change'].apply(
        lambda x: pd.Series(split_aa_change(x))
    )

# Check a few non-null aa_change rows
print(df_b1[df_b1['aa_change'].notna()][
    ['vid', 'consequence', 'aa_change', 'protein_id', 'hgvsp']
].head(10).to_string())

#No unique identifiers present for participant data

                 vid         consequence                              aa_change         protein_id                hgvsp
276  13-32316463-G-A          start_lost                  ENSP00000369497.3:p.?  ENSP00000369497.3                  p.?
277  13-32316467-A-G    missense_variant          ENSP00000369497.3:p.(Ile3Val)  ENSP00000369497.3          p.(Ile3Val)
278  13-32316489-C-A    missense_variant         ENSP00000369497.3:p.(Thr10Lys)  ENSP00000369497.3         p.(Thr10Lys)
279  13-32316511-A-G  synonymous_variant  ENST00000380152.7:c.51A>G(p.(Thr17=))  ENST00000380152.7  c.51A>G(p.(Thr17=))
280  13-32316513-G-A    missense_variant         ENSP00000369497.3:p.(Arg18His)  ENSP00000369497.3         p.(Arg18His)
281  13-32316522-A-G    missense_variant         ENSP00000369497.3:p.(Lys21Arg)  ENSP00000369497.3         p.(Lys21Arg)
286  13-32319088-A-G    missense_variant         ENSP00000369497.3:p.(Ile27Val)  ENSP00000369497.3         p.(Ile27Val)
287  13-32319091-A-G    missense_variant

In [6]:
print("=== Consequence counts ===")
print(df_b1['consequence'].value_counts())

print("\n=== ClinVar classification counts ===")
print(df_b1['clinvar_classification'].fillna('None').value_counts())

print("\n=== gvs_max_subpop distribution ===")
print(df_b1['gvs_max_subpop'].value_counts())

print("\n=== gnomad_max_subpop distribution (including NaN) ===")
print(df_b1['gnomad_max_subpop'].value_counts(dropna=False))


=== Consequence counts ===
consequence
missense_variant                              888
downstream_gene_variant                       373
synonymous_variant                            312
3_prime_UTR_variant                           232
upstream_gene_variant                         204
frameshift_variant                             68
5_prime_UTR_variant                            64
intron_variant                                 54
intron_variant, splice_region_variant          35
stop_gained                                    31
inframe_deletion                               24
missense_variant, splice_region_variant        18
splice_region_variant, synonymous_variant       6
splice_acceptor_variant                         5
inframe_insertion                               3
5_prime_UTR_variant, splice_region_variant      2
start_lost                                      1
splice_region_variant, stop_gained              1
frameshift_variant, stop_gained                 1
Name: count

In [7]:
OUT_FILE = "2026_B2_variants_amr_parsed_CT.xlsx"

export_cols = [
    # 1) Hail / QC / info (before old column F)
    'locus', 'alleles',
    'qc_AC', 'qc_AF', 'qc_AN',
    'call_rate', 'n_het',
    'info_AC', 'info_AF', 'info_AN',

    # 2) Core VAT annotation (as in 2023 BRCA2)
    'vid', 'contig', 'position', 'consequence', 'clinvar_classification',
    'variant_type', 'ref_allele', 'alt_allele',
    'gvs_all_ac', 'gvs_all_an', 'gvs_all_af', 'gvs_all_sc',
    'dbsnp_rsid', 'revel',
    'vid_alt',        # present in both years

    # 3) 8 new VAT variables in 2026 BRCA1
    'gvs_max_subpop', 'aa_change', 'transcript', 'gene_symbol',
    'gvs_max_ac', 'gvs_max_sc', 'gvs_max_an', 'gnomad_max_subpop',

    # 4) Derived from aa_change
    'protein_id', 'hgvsp'
]

# Keep only columns that actually exist, to avoid KeyError
export_cols = [c for c in export_cols if c in df_b1.columns]

df_b1[export_cols].to_excel(OUT_FILE, index=False)
print(f"Exported {len(df_b1)} rows × {len(export_cols)} columns → {OUT_FILE}")


Exported 2322 rows × 35 columns → 2026_B2_variants_amr_parsed_CT.xlsx


______________________________________________________________________________

## older sheets cleaning below

a. reads Excel,

b. reconstructs the “cramped” annotation text that got split across column F (and sometimes into G),

c. parses it into sortable columns like consequence, clinvar_classification, and revel.

In [ ]:
pip install openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\farwa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip




   ---------------------------------------- 0/2 [et-xmlfile]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [ope

In [ ]:
import pandas as pd
import numpy as np
import re
import shlex

FILE = "New BRCA2 Variants by CK 2-20-26.xlsx"

# Inspect sheets
xls = pd.ExcelFile(FILE)
xls.sheet_names


['New BRCA2 Variants by CK 2-20-2', 'ROW FIELDS Requested']

In [ ]:
FILE2026 = "New BRCA2 Variants by CK 2-20-26.xlsx"

# Inspect sheets
xls = pd.ExcelFile(FILE2026)
xls.sheet_names



['New BRCA2 Variants by CK 2-20-2', 'ROW FIELDS Requested']

In [ ]:
# Read the main sheet with no header so we can handle the nonstandard header rows
SHEET_MAIN = xls.sheet_names[0]
raw = pd.read_excel(FILE2026, sheet_name=SHEET_MAIN, header=None)

raw.shape, raw.head(12)

#Data preview removed to comply with All of Us data user policies.


concatenating F and G (when G is actually a continuation), then normalizing quotes/whitespace.

In [ ]:
# Identify data rows: in file, variant records begin where col0 starts with 'chr13:' (e.g., chr13:32310629 ...)
data_mask = raw[0].astype(str).str.startswith("chr13:", na=False)
data = raw.loc[data_mask].copy()

# Grab F and G text
colF = data[5].fillna("").astype(str)
colG = data[6].fillna("").astype(str)

# Combine: many broken rows have consequence/clinvar split across F and G
combined = (colF + " " + colG).str.strip()

# Clean leading artifacts like 0]" that appear before the real content in F
combined = combined.str.replace(r'^\s*\d+\]\\"\s*', '', regex=True)
combined = combined.str.replace(r'^\s*\d+\]\"\s*', '', regex=True)

# Normalize whitespace
combined = combined.str.replace(r"\s+", " ", regex=True).str.strip()

data["combined_payload"] = combined
data[["combined_payload"]].head(10)

#Data preview removed to comply with All of Us data user policies.

### Parse into separate columns (sortable)
The payload is mostly space-delimited, but clinvar_classification can be a quoted 
multi-word phrase (e.g., "likely benign", "uncertain significance"), so shlex.split() 
is a good fit

In [ ]:
import re
import shlex

VARIANT_TYPES = {"SNV", "deletion", "insertion"}

# Patterns based on what appears in sheet
VID_RE = re.compile(r"^(?:[0-9]+|X|Y|M|MT)-\d+-\S+-\S+$")  # e.g., 13-32311358-T-C, 13-32310629-TTTTA-T
VID_ALT_RE = re.compile(r"^chr\w+:\d+:[^:\s]+:[^:\s]+$")   # e.g., chr13:32311358:T:C
CONTIG_RE = re.compile(r"^chr\w+$")                        # e.g., chr13
POS_RE = re.compile(r"^\d+$")                              # position token

def parse_payload(s: str) -> dict:
    # tokenize (preserve quoted strings when possible)
    try:
        toks = shlex.split(s)
    except ValueError:
        toks = str(s).split()

    out = {
        "vid": None,
        "contig": None,
        "position": None,
        "consequence": None,
        "clinvar_classification": None,
        "variant_type": None,
        "ref_allele": None,
        "alt_allele": None,
        "gvs_all_ac": None,
        "gvs_all_an": None,
        "gvs_all_af": None,
        "gvs_all_sc": None,
        "dbsnp_rsid": None,
        "revel": None,
        "vid_alt": None,
        "tokens_n": len(toks),
    }
    if len(toks) < 6:
        return out

    # --- Find vid by pattern (don’t assume toks[0]) ---
    vid_idx = next((i for i, t in enumerate(toks) if VID_RE.match(t)), None)
    if vid_idx is None:
        return out
    out["vid"] = toks[vid_idx]

    # contig and position (prefer immediately after vid; otherwise search forward)
    contig_idx = vid_idx + 1 if (vid_idx + 1 < len(toks) and CONTIG_RE.match(toks[vid_idx + 1])) \
                else next((i for i in range(vid_idx + 1, len(toks)) if CONTIG_RE.match(toks[i])), None)
    if contig_idx is None:
        return out
    out["contig"] = toks[contig_idx]

    pos_idx = contig_idx + 1 if (contig_idx + 1 < len(toks) and POS_RE.match(toks[contig_idx + 1])) \
              else next((i for i in range(contig_idx + 1, len(toks)) if POS_RE.match(toks[i])), None)
    if pos_idx is None:
        return out
    out["position"] = toks[pos_idx]

    # consequence is the next token after position (as in your row fields)
    conseq_idx = pos_idx + 1
    if conseq_idx >= len(toks):
        return out
    out["consequence"] = toks[conseq_idx]

    i = conseq_idx + 1

    # clinvar_classification (optional)
    if i < len(toks) and toks[i] not in VARIANT_TYPES:
        out["clinvar_classification"] = toks[i]
        i += 1

    # variant_type/ref/alt
    if i < len(toks): out["variant_type"] = toks[i]; i += 1
    if i < len(toks): out["ref_allele"] = toks[i]; i += 1
    if i < len(toks): out["alt_allele"] = toks[i]; i += 1

    def to_int(x):
        try: return int(x)
        except: return None

    def to_float(x):
        try: return float(x)
        except: return None

    if i < len(toks): out["gvs_all_ac"] = to_int(toks[i]); i += 1
    if i < len(toks): out["gvs_all_an"] = to_int(toks[i]); i += 1
    if i < len(toks): out["gvs_all_af"] = to_float(toks[i]); i += 1
    if i < len(toks): out["gvs_all_sc"] = to_int(toks[i]); i += 1

    if i < len(toks) and str(toks[i]).startswith("rs"):
        out["dbsnp_rsid"] = toks[i]; i += 1

    if i < len(toks):
        maybe_revel = to_float(toks[i])
        if maybe_revel is not None:
            out["revel"] = maybe_revel
            i += 1

    # --- vid_alt by pattern anywhere in toks (prevents 'Ensembl' contamination) ---
    vid_alt_hits = [t for t in toks if VID_ALT_RE.match(t)]
    out["vid_alt"] = vid_alt_hits[-1] if vid_alt_hits else None

    return out

parsed = data["combined_payload"].apply(parse_payload).apply(pd.Series)
parsed["excel_row"] = data.index + 1
parsed.head()

#Data preview removed to comply with All of Us data user policies.

In [ ]:
parsed.head(15)

#Data preview removed to comply with All of Us data user policies.

creates quick sanity checks for missing ClinVar significance and flags “REVEL present on upstream variants"

In [ ]:
# How many are missing ClinVar?
parsed["clinvar_classification"].isna().mean(), parsed["clinvar_classification"].value_counts(dropna=False).head(20)


(0.49162011173184356,
 clinvar_classification
 None                                        174
 likely benign uncertain significance         68
 uncertain significance                       41
 likely benign                                37
 pathogenic                                   22
 likely pathogenic pathogenic                  6
 splice_region_variant"                        2
 non_coding_transcript_variant"                2
 NaN                                           2
 benign likely benign                          1
 uncertain significance likely pathogenic      1
 missense_variant"                             1
 NMD_transcript_variant"                       1
 Name: count, dtype: int64)

In [ ]:
# Consequence distribution (can sort/pull missense easily)
parsed["consequence"].value_counts().head(25)


consequence
downstream_gene_variant                                     99
missense_variant                                            78
non_coding_transcript_exon_variant                          57
upstream_gene_variant                                       36
synonymous_variant                                          25
frameshift_variant                                          14
3_prime_UTR_variant                                         12
stop_gained                                                  7
5_prime_UTR_variant                                          4
intron_variant                                               4
"intron_variant                                              3
missense_variant splice_region_variant                       3
inframe_deletion                                             2
intron_variant splice_region_variant                         2
NMD_transcript_variant missense_variant                      2
"non_coding_transcript_exon_variant        

In [ ]:
# View the rows referenced (Excel rows 10 and 74)
parsed.loc[parsed["excel_row"].isin([10, 74]),
           ["excel_row","vid","consequence","clinvar_classification","variant_type","dbsnp_rsid","revel","vid_alt"]]

#No unique identifiers present for participant data

,excel_row,vid,consequence,clinvar_classification,variant_type,dbsnp_rsid,revel,vid_alt
9,10,13-32311358-T-C,upstream_gene_variant,likely benign,SNV,rs1024981194,0.084,chr13:32311358:T:C
73,74,13-32332665-C-T,missense_variant,likely benign uncertain significance,SNV,rs2137468510,0.184,chr13:32332665:C:T


In [ ]:
# Flag upstream_gene_variant entries that nonetheless have a REVEL score filled
flag = parsed[(parsed["consequence"] == "upstream_gene_variant") & (parsed["revel"].notna())]
flag[["excel_row","vid","consequence","clinvar_classification","revel","vid_alt"]].head(20)

#No unique identifiers present for participant data

,excel_row,vid,consequence,clinvar_classification,revel,vid_alt
4,5,13-32310658-C-G,upstream_gene_variant,None,0.354,chr13:32310658:C:G
6,7,13-32311327-C-T,upstream_gene_variant,None,0.042,chr13:32311327:C:T
8,9,13-32311348-G-C,upstream_gene_variant,None,0.015,chr13:32311348:G:C
9,10,13-32311358-T-C,upstream_gene_variant,likely benign,0.084,chr13:32311358:T:C
10,11,13-32311366-G-A,upstream_gene_variant,None,0.011,chr13:32311366:G:A
11,12,13-32311384-G-A,upstream_gene_variant,None,0.011,chr13:32311384:G:A
12,13,13-32311421-G-A,upstream_gene_variant,None,0.053,chr13:32311421:G:A
13,14,13-32311489-C-T,upstream_gene_variant,None,0.030,chr13:32311489:C:T
14,15,13-32311560-G-T,upstream_gene_variant,None,0.050,chr13:32311560:G:T
15,16,13-32311611-G-C,upstream_gene_variant,None,0.293,chr13:32311611:G:C


## Export clean table

In [ ]:
OUT_XLSX = "2026_New_BRCA2_variants_cleaned.xlsx"

parsed.to_excel(OUT_XLSX, index=False)

OUT_XLSX


'New_BRCA2_variants_cleaned.xlsx'

# 2023 File Splitting

In [ ]:
FILE2023 = "2023_B2_variants_amr.csv"

# Read the CSV file with no header so we can handle the nonstandard header rows
raw = pd.read_csv(FILE2023, header=None, sep='\t')

raw.shape, raw.head(12)

#Data preview removed to comply with All of Us data user policies.

In [ ]:
import pandas as pd

def parse_space_delimited_line(line):
    """
    Parse a space-delimited line with quoted fields.
    Empty fields (consecutive spaces) are preserved as empty strings.
    Quoted fields (e.g., "['A', 'G']") are handled correctly.
    """
    fields = []
    i = 0
    line = line.rstrip('\n')
    while i < len(line):
        if line[i] == '"':
            j = line.index('"', i + 1)
            fields.append(line[i+1:j])
            i = j + 1
            if i < len(line) and line[i] == ' ':
                i += 1
        elif line[i] == ' ':
            fields.append('')   # empty field from consecutive spaces
            i += 1
        else:
            j = line.find(' ', i)
            if j == -1:
                fields.append(line[i:])
                break
            fields.append(line[i:j])
            i = j + 1
    return fields

# --- Parse file ---
with open('2023_B2_variants_amr.csv', 'r') as f:
    lines = f.readlines()

header = lines[0].rstrip('\n').split(' ')
rows = [parse_space_delimited_line(l) for l in lines[1:] if l.strip()]
df = pd.DataFrame(rows, columns=header)

# --- Select & rename key columns ---
keep_cols = [
    'locus', 'alleles',
    'vat.vid', 'vat.contig', 'vat.position',
    'vat.consequence', 'vat.clinvar_classification', 'vat.variant_type',
    'vat.ref_allele', 'vat.alt_allele',
    'vat.gvs_all_ac', 'vat.gvs_all_an', 'vat.gvs_all_af',
    'vat.dbsnp_rsid', 'vat.revel', 'vat.vid_alt',
    'variant_qc.AC', 'variant_qc.AF', 'variant_qc.AN',
    'variant_qc.call_rate', 'variant_qc.n_het',
    'info.AC', 'info.AF', 'info.AN'
]
df_clean = df[keep_cols].copy()
df_clean.replace('', pd.NA, inplace=True)

rename_map = {
    'vat.vid': 'vid', 'vat.contig': 'contig', 'vat.position': 'position',
    'vat.consequence': 'consequence', 'vat.clinvar_classification': 'clinvar_classification',
    'vat.variant_type': 'variant_type', 'vat.ref_allele': 'ref_allele',
    'vat.alt_allele': 'alt_allele', 'vat.gvs_all_ac': 'gvs_all_ac',
    'vat.gvs_all_an': 'gvs_all_an', 'vat.gvs_all_af': 'gvs_all_af',
    'vat.dbsnp_rsid': 'dbsnp_rsid', 'vat.revel': 'revel', 'vat.vid_alt': 'vid_alt',
    'variant_qc.AC': 'qc_AC', 'variant_qc.AF': 'qc_AF', 'variant_qc.AN': 'qc_AN',
    'variant_qc.call_rate': 'call_rate', 'variant_qc.n_het': 'n_het',
    'info.AC': 'info_AC', 'info.AF': 'info_AF', 'info.AN': 'info_AN'
}
df_clean.rename(columns=rename_map, inplace=True)

# --- Export to Excel with auto-filter + frozen header ---
with pd.ExcelWriter('2023_B2_variants_amr_cleaned.xlsx', engine='openpyxl') as writer:
    df_clean.to_excel(writer, index=False, sheet_name='AMR_Variants_Cleaned')
    ws = writer.sheets['AMR_Variants_Cleaned']
    ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = 'A2'
    for col_cells in ws.columns:
        max_len = max((len(str(c.value)) if c.value else 0) for c in col_cells)
        ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 40)


In [ ]:
#number of missing ClinVar classifications in the cleaned CSV
missing_clinvar = df_clean['clinvar_classification'].isna().sum()

In [ ]:
#print the result
print(f"Number of missing ClinVar classifications in the cleaned CSV: {missing_clinvar}")

Number of missing ClinVar classifications in the cleaned CSV: 638
